In [65]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [66]:
### core fundataion - llm
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

### Node 1: JD Parser

In [120]:

from pydantic import BaseModel
from typing import List

class JDStructured(BaseModel):
    role: str
    location: str
    experience_years: int
    required_skills: List[str]
    preferred_skills: List[str]
    domain: str
    seniority: str

In [121]:
from langchain_core.prompts import ChatPromptTemplate

jd_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are an expert recruiter who extracts structured information from messy job descriptions.

Return clean structured JSON only.
"""),

    ("user", """
Extract the following from this Job Description:

- role
- location
- experience_years (number)
- required_skills (list)
- preferred_skills (list)
- domain (industry/domain)
- seniority (junior/mid/senior)

Job Description:
{jd}
""")
])

In [123]:
structured_llm = llm.with_structured_output(JDStructured)

chain = jd_prompt | structured_llm

In [124]:
jd_text = """
Job description
Position: Data ScientistLocation: Hyderabad
Industry: IT Services & Consulting
Department: Engineering Software & QA
Employment Type: Full-Time, Permanent


Job Description We are seeking a Data Scientist to design, build, and deploy machine learning models and AI-powered solutions across manufacturing, supply chain, and engineering. You will develop predictive models, build AI agents using LLMs and RAG architectures, and deliver insights that drive yield improvement, process optimization, and intelligent automation.


Responsibilities

Build and deploy ML models for yield prediction, defect classification, and demand forecasting
Develop AI agents using LLMs, RAG pipelines, and multi-agent frameworks
Engineer end-to-end data pipelines for model training, inference, and monitoring
Partner with cross-functional teams to translate business problems into ML solutions
Deliver actionable insights through dashboards and conversational AI interfaces

Required Qualifications

BS/MS in Data Science, Computer Science, Statistics, or related field
2+ years building and deploying ML models in production
Strong Python (scikit-learn, pandas, PyTorch/TensorFlow) and SQL
Experience with LLM integration — prompt engineering, fine-tuning, or RAG
Solid communication skills with technical and non-technical audiences

Preferred Qualifications

Semiconductor or manufacturing industry experience
Hands-on with Dataiku, Azure ML, or Snowflake Cortex
Experience with knowledge graphs, AI agents, or multi-agent systems

Skills

ML / Data Science
Classification & Regression
Time-Series Forecasting
Anomaly Detection
Feature Engineering
Model Explainability (SHAP)

AI / LLM

Prompt Engineering
Retrieval-Augmented Generation (RAG)
Fine-Tuning
AI Agent Design
Knowledge Graphs & Embeddings

Programming & Platforms

Python (scikit-learn, pandas, PyTorch/TensorFlow)
SQL (Snowflake)
REST APIs & Git
Dataiku
Azure ML & Azure OpenAI
Power BI

Role: Data Scientist
Industry Type: IT Services & Consulting
Department: Data Science & Analytics
Employment Type: Full Time, Permanent
Role Category: Data Science & Machine Learning
Education
UG: Any Graduate
Key Skills
Skills highlighted with ‘‘ are preferred keyskills
Machine Learning
Data ScienceDeep LearningPython

"""

In [125]:


jd_parsed = chain.invoke({"jd": jd_text})

print(jd_parsed)

role='Data Scientist' location='Hyderabad' experience_years=2 required_skills=['Python (scikit-learn, pandas, PyTorch/TensorFlow)', 'SQL', 'Machine Learning', 'Data Science'] preferred_skills=['Semiconductor or manufacturing industry experience', 'Hands-on with Dataiku, Azure ML, or Snowflake Cortex', 'Experience with knowledge graphs, AI agents, or multi-agent systems'] domain='IT Services & Consulting' seniority='mid'


In [126]:
jd_parsed

JDStructured(role='Data Scientist', location='Hyderabad', experience_years=2, required_skills=['Python (scikit-learn, pandas, PyTorch/TensorFlow)', 'SQL', 'Machine Learning', 'Data Science'], preferred_skills=['Semiconductor or manufacturing industry experience', 'Hands-on with Dataiku, Azure ML, or Snowflake Cortex', 'Experience with knowledge graphs, AI agents, or multi-agent systems'], domain='IT Services & Consulting', seniority='mid')

### Node 2: Candidate Retriever

In [128]:
candidate_pool = pd.read_csv("candidate database.csv", encoding="latin1")

In [131]:
import pandas as pd

df = candidate_pool.copy()

# Group by candidate
grouped = df.groupby("id").agg({
    "candidate_name": "first",
    "skill_name": lambda x: list(set([s.lower() for s in x.dropna()])),
    "certification_name": lambda x: list(set(x.dropna()))
}).reset_index()

In [133]:
import random

def generate_experience():
    return random.randint(1, 6)

def generate_role(skills):
    if "machine learning" in skills or "deep learning" in skills:
        return "Data Scientist"
    elif "sql" in skills:
        return "Data Analyst"
    else:
        return "Software Engineer"

In [134]:
def build_summary(skills, certs):
    return f"Skilled in {', '.join(skills[:5])}. Certifications: {', '.join(certs[:3])}"

In [135]:
candidates = []

for _, row in grouped.iterrows():

    skills = row["skill_name"]
    certs = row["certification_name"]

    candidate = {
        "name": row["candidate_name"],
        "skills": skills,
        "experience": generate_experience(),
        "summary": build_summary(skills, certs),
        "current_role": generate_role(skills)
    }

    candidates.append(candidate)

### create the candidate profile

candidates = [
    {
        "id": 1,
        "name": "Rahul Sharma",
        "skills": ["ML","Python","SQL"],
        "experience": 3,
        "location": "Bangalore",
        "summary": "Data Scientist with ML and analytics experience"
    },
    {
        "id": 2,
        "name": "Ankit Verma",
        "skills": ["Java", "Spring", "AWS"],
        "experience": 5,
        "location": "Delhi",
        "summary": "Backend engineer with cloud expertise"
    },
    {
        "id": 3,
        "name": "Priya Singh",
        "skills": ["Python", "Deep Learning", "NLP"],
        "experience": 4,
        "location": "Bangalore",
        "summary": "AI Engineer specializing in NLP"
    }
]

In [117]:
import json
from functools import lru_cache
from typing import List

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field


# LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


# ✅ FIXED SCHEMA (List-based, NOT dict)
class SkillItem(BaseModel):
    skill: str
    synonyms: List[str]


class SkillList(BaseModel):
    skills: List[SkillItem]


# Prompt
skill_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a recruiting expert. Return ONLY valid JSON."),
    ("user", """
Expand each skill into synonyms, abbreviations, and closely related terms.

Role: {role}
Skills: {skills}

Rules:
- Include original skill
- Include abbreviations (ML, NLP, etc.)
- Include tools/frameworks
- Max 5–7 per skill

Return JSON like:
{{
  "skills": [
    {{"skill": "python", "synonyms": ["python", "py"]}},
    {{"skill": "machine learning", "synonyms": ["ml", "machine learning"]}}
  ]
}}
""")
])


# Chain
skill_chain = skill_prompt | llm.with_structured_output(SkillList)


# Cached function
@lru_cache(maxsize=32)
def expand_skills_with_llm(required_skills_tuple: tuple, role: str = ""):
    required_skills = list(required_skills_tuple)

    result = skill_chain.invoke({
        "role": role,
        "skills": json.dumps(required_skills)
    })

    # ✅ Convert list → dict (your expected format)
    skill_map = {}

    for item in result.skills:
        key = item.skill.lower().strip()
        synonyms = list(set([s.lower().strip() for s in item.synonyms]))
        skill_map[key] = synonyms

    return skill_map

In [118]:
import re
import json
from typing import List, Dict

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

from sklearn.metrics.pairwise import cosine_similarity

#from nodes.skill_expander import expand_skills_with_llm


# LLM + Embeddings
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


# -------------------------------
# Helpers
# -------------------------------
def _normalize(text: str) -> str:
    return re.sub(r"\s+", " ", text.lower().strip())


# -------------------------------
# 1. KEYWORD FILTER
# -------------------------------
def keyword_prefilter(
    candidates: List[Dict],
    required_skills: List[str],
    skill_synonyms: Dict[str, List[str]],
    min_experience: int = 0,
    skill_match_threshold: float = 0.5,
) -> List[Dict]:

    shortlisted = []

    for candidate in candidates:

        if candidate.get("experience", 0) < min_experience:
            continue

        candidate_text = _normalize(
            " ".join(candidate.get("skills", []))
            + " " + candidate.get("summary", "")
        )

        matched_skills = []

        for req_skill in required_skills:
            skill_key = req_skill.lower().strip()
            synonyms = skill_synonyms.get(skill_key, [skill_key])

            pattern = r"\b(" + "|".join(re.escape(s) for s in synonyms) + r")\b"

            if re.search(pattern, candidate_text):
                matched_skills.append(req_skill)

        match_ratio = len(matched_skills) / len(required_skills) if required_skills else 0

        if match_ratio >= skill_match_threshold:
            shortlisted.append({
                **candidate,
                "_keyword_matched_skills": matched_skills,
                "_keyword_match_ratio": round(match_ratio, 2),
            })

    return shortlisted


# -------------------------------
# 2. EMBEDDING RERANK
# -------------------------------
def embedding_rerank(candidates, jd_parsed, top_n=10):

    jd_text = f"""
    Role: {jd_parsed.get('role')}
    Skills: {', '.join(jd_parsed.get('required_skills', []))}
    Experience: {jd_parsed.get('experience_years')}
    """

    jd_embedding = embeddings.embed_query(jd_text)

    for candidate in candidates:
        cand_text = f"""
        Skills: {', '.join(candidate.get('skills', []))}
        Experience: {candidate.get('experience')}
        Summary: {candidate.get('summary')}
        """

        cand_embedding = embeddings.embed_query(cand_text)

        score = cosine_similarity(
            [jd_embedding],
            [cand_embedding]
        )[0][0]

        candidate["_embedding_score"] = round(score * 100, 1)

    return sorted(candidates, key=lambda x: x["_embedding_score"], reverse=True)[:top_n]


# -------------------------------
# 3. LLM SCORING
# -------------------------------
class CandidateScore(BaseModel):
    match_score: int
    matched_skills: List[str]
    missing_skills: List[str]
    explanation: str


scoring_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a strict recruiter. Score candidate vs JD."),
    ("user", """
JD:
{jd}

Candidate:
{candidate}

Return:
- match_score (0-100)
- matched_skills
- missing_skills
- explanation (2 lines)
""")
])


scoring_chain = scoring_prompt | llm.with_structured_output(CandidateScore)


def llm_score(candidates, jd_parsed):

    scored = []

    for candidate in candidates:

        result = scoring_chain.invoke({
            "jd": json.dumps(jd_parsed),
            "candidate": json.dumps(candidate)
        })

        scored.append({
            **candidate,
            "match_score": result.match_score,
            "matched_skills": result.matched_skills,
            "missing_skills": result.missing_skills,
            "match_explanation": result.explanation,
        })

    return sorted(scored, key=lambda x: x["match_score"], reverse=True)


# -------------------------------
# 4. PIPELINE (Jupyter)
# -------------------------------
def run_candidate_pipeline(jd_parsed, candidates):

    # FIX: convert Pydantic → dict
    if not isinstance(jd_parsed, dict):
        jd_parsed = jd_parsed.model_dump()

    print(f"[filter] Total candidates: {len(candidates)}")

    # Step 1 — Skill Expansion
    skill_synonyms = expand_skills_with_llm(
        tuple(jd_parsed.get("required_skills", [])),
        role=jd_parsed.get("role", "")
    )

    print("[filter] Skill Synonyms:", json.dumps(skill_synonyms, indent=2))

    # Step 2 — Keyword filter
    prefiltered = keyword_prefilter(
        candidates=candidates,
        required_skills=jd_parsed.get("required_skills", []),
        skill_synonyms=skill_synonyms,
        min_experience=jd_parsed.get("experience_years", 0) - 1,
        skill_match_threshold=0.5,
    )

    print(f"[filter] After keyword filter: {len(prefiltered)}")

    # Step 3 — Embedding rerank (conditional)
    if len(prefiltered) > 5:
        reranked = embedding_rerank(prefiltered, jd_parsed)
    else:
        reranked = prefiltered

    print(f"[filter] After rerank: {len(reranked)}")

    # Step 4 — LLM scoring
    scored = llm_score(reranked, jd_parsed)

    print(f"[filter] Final candidates: {len(scored)}")

    return {
        "scored_candidates": scored,
        "skill_synonyms": skill_synonyms
    }

In [137]:
result = run_candidate_pipeline(jd_parsed, candidates)

for c in result["scored_candidates"]:
    print(c["name"], c["match_score"])

[filter] Total candidates: 936
[filter] Skill Synonyms: {
  "python (scikit-learn, pandas, pytorch/tensorflow)": [
    "pytorch",
    "pandas",
    "py",
    "tensorflow",
    "numpy",
    "python",
    "scikit-learn"
  ],
  "sql": [
    "pl/sql",
    "postgresql",
    "sql",
    "structured query language",
    "sqlite",
    "t-sql",
    "mysql"
  ],
  "machine learning": [
    "deep learning",
    "nlp",
    "reinforcement learning",
    "supervised learning",
    "machine learning",
    "ml",
    "unsupervised learning"
  ],
  "data science": [
    "data science",
    "predictive analytics",
    "big data",
    "data analytics",
    "data analysis",
    "statistical analysis",
    "data mining"
  ]
}
[filter] After keyword filter: 35
[filter] After rerank: 10
[filter] Final candidates: 10
Rahul Mohalder 85
Md. Mostafa 85
Asif Faisal 75
Monirul Khan 70
Md. Minhaz Ul Karim . 60
Abu Jubair Nayeem 60
Mohammad Hoque 50
Md.Nur Khan 50
Kalyan Banik 50
Md Shahinur Rahman 40


### Node 4: Outreach Agent

In [138]:
from pydantic import BaseModel

class CandidateReply(BaseModel):
    reply: str

class InterestScore(BaseModel):
    interest_score: int
    explanation: str

In [139]:
from langchain_core.prompts import ChatPromptTemplate

reply_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a job candidate responding to a recruiter.

Be realistic:
- Some candidates are actively looking
- Some are passive
- Some are not interested

Keep response natural and human-like.
"""),

    ("user", """
Candidate Profile:
{candidate}

Recruiter Message:
{message}

Reply as the candidate.
""")
])

In [140]:
score_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are an expert recruiter.

Based on the conversation, estimate candidate interest.

Return:
- interest_score (0-100)
- explanation (1-2 lines)

Guidelines:
0-30 → Not interested  
30-60 → Passive  
60-100 → Interested  
"""),

    ("user", """
Conversation:
{conversation}
""")
])

In [141]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

reply_chain = reply_prompt | llm.with_structured_output(CandidateReply)
score_chain = score_prompt | llm.with_structured_output(InterestScore)

In [142]:
import random

def run_outreach(state: dict) -> dict:

    jd = state["jd_parsed"]
    if not isinstance(jd, dict):
        jd = jd.model_dump()

    for sc in state["scored_candidates"]:

        # Skip low matches
        if sc["match_score"] < 40:
            sc["interest_score"] = 0
            sc["interest_explanation"] = "Low match score"
            sc["conversation_log"] = []
            sc["response_status"] = "Not Contacted"
            continue

        candidate = sc

        # -------------------------------
        # Turn 1 — Agent Message
        # -------------------------------
        opening = f"""
Hi {candidate.get('name')},

We have a {jd.get('seniority', 'mid-level')} role in {jd.get('domain', 'data science')}.

Your experience in {', '.join(candidate.get('skills', [])[:2])} looks like a strong fit.

Would you be open to exploring this opportunity?
"""

        # -------------------------------
        # NEW: Simulate No Response
        # -------------------------------
        no_response_prob = 0.2   # 20% ghost rate

        if random.random() < no_response_prob:
            sc["interest_score"] = 0
            sc["interest_explanation"] = "Candidate did not respond"
            sc["conversation_log"] = [
                {"role": "agent", "content": opening}
            ]
            sc["response_status"] = "No Response"

            sc["final_score"] = round(
                0.7 * sc["match_score"], 2
            )
            continue

        # -------------------------------
        # Turn 2 — Candidate Reply
        # -------------------------------
        reply_obj = reply_chain.invoke({
            "candidate": json.dumps(candidate),
            "message": opening
        })

        reply = reply_obj.reply

        conversation = [
            {"role": "agent", "content": opening},
            {"role": "candidate", "content": reply}
        ]

        # -------------------------------
        # Turn 3 — Interest Score
        # -------------------------------
        score_obj = score_chain.invoke({
            "conversation": json.dumps(conversation)
        })

        sc["interest_score"] = score_obj.interest_score
        sc["interest_explanation"] = score_obj.explanation
        sc["conversation_log"] = conversation
        sc["response_status"] = "Responded"

        sc["final_score"] = round(
            0.7 * sc["match_score"] + 0.3 * sc["interest_score"], 2
        )

    # Sort
    state["scored_candidates"] = sorted(
        state["scored_candidates"],
        key=lambda x: x.get("final_score", 0),
        reverse=True
    )

    return state

In [143]:
state = {
    "jd_parsed": jd_parsed,
    "scored_candidates": result["scored_candidates"]
}

state = run_outreach(state)

for c in state["scored_candidates"]:
    print(c["name"])
    print("Match:", c["match_score"])
    print("Interest:", c["interest_score"])
    print("Final:", c["final_score"])
    print("Status:", c.get("response_status"))

    reply = (
        c["conversation_log"][1]["content"]
        if len(c.get("conversation_log", [])) > 1
        else "No response"
    )

    print("Reply:", reply)
    print("-" * 50)

Md. Mostafa
Match: 85
Interest: 75
Final: 82.0
Status: Responded
Reply: Hi [Recruiter's Name],

Thank you for reaching out! I appreciate you considering my experience for the mid role in IT Services & Consulting. 

I am currently exploring new opportunities, and the position sounds interesting. However, I noticed that the role may require some experience in the semiconductor or manufacturing industry, as well as familiarity with Dataiku, Azure ML, or Snowflake Cortex, which I do not possess at the moment. 

That said, I'd be happy to learn more about the role and the team. Could we possibly schedule a call to discuss this opportunity further?

Thanks again!

Best regards,
Md. Mostafa
--------------------------------------------------
Rahul Mohalder
Match: 85
Interest: 70
Final: 80.5
Status: Responded
Reply: Hi [Recruiter's Name],

Thank you for reaching out and considering me for the mid role in IT Services & Consulting. I appreciate your kind words about my experience and skills in da

### Node : Reranker

In [144]:
from pydantic import BaseModel

class ConversationSummary(BaseModel):
    summary: str

In [145]:
from langchain_core.prompts import ChatPromptTemplate

summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "You summarize recruiter-candidate conversations."),
    ("user", """
Conversation:
{conversation}

Summarize in 1-2 lines focusing on:
- candidate interest
- any concerns
""")
])

In [146]:
summary_chain = summary_prompt | llm.with_structured_output(ConversationSummary)

In [147]:
def assign_status(final_score):
    if final_score >= 85:
        return "High Priority"
    elif final_score >= 65:
        return "Consider"
    else:
        return "Low Priority"

In [148]:
def rerank_and_format(state: dict) -> dict:

    candidates = state["scored_candidates"]
    jd = state["jd_parsed"]

    if not isinstance(jd, dict):
        jd = jd.model_dump()

    final_output = []

    for c in candidates:

        # -------------------------------
        # Scores
        # -------------------------------
        match_score = c.get("match_score", 0)
        interest_score = c.get("interest_score", 0)

        final_score = round(
            0.6 * match_score + 0.4 * interest_score,
            2
        )

        # -------------------------------
        # Conversation Summary
        # -------------------------------
        conversation = c.get("conversation_log", [])

        if conversation:
            summary_obj = summary_chain.invoke({
                "conversation": str(conversation)
            })
            summary = summary_obj.summary
        else:
            summary = "No interaction"

        # -------------------------------
        # Explanation
        # -------------------------------
        explanation = f"""
Matched Skills: {', '.join(c.get('matched_skills', []))}
Missing Skills: {', '.join(c.get('missing_skills', []))}
Candidate Interest Score: {interest_score}
"""

        # -------------------------------
        # Final Object (UPDATED)
        # -------------------------------
        final_output.append({
            "candidate_name": c.get("name"),
            "match_score": match_score,
            "interest_score": interest_score,
            "final_score": final_score,
            "status": assign_status(final_score),
            "explanation": explanation.strip(),
            "conversation_summary": summary
        })

    # Sort
    final_output = sorted(
        final_output,
        key=lambda x: x["final_score"],
        reverse=True
    )

    state["final_output"] = final_output
    return state

In [149]:
state = run_outreach(state)
state = rerank_and_format(state)

for c in state["final_output"]:
    print("⭐", c["candidate_name"])
    print("Match Score:", c["match_score"])
    print("Interest Score:", c["interest_score"])
    print("Final Score:", c["final_score"])
    print("Status:", c["status"])
    print("Explanation:", c["explanation"])
    print("Summary:", c["conversation_summary"])
    print("=" * 60)

⭐ Md. Mostafa
Match Score: 85
Interest Score: 75
Final Score: 81.0
Status: Consider
Explanation: Matched Skills: Python (scikit-learn, pandas, PyTorch/TensorFlow), SQL, Machine Learning, Data Science
Missing Skills: Semiconductor or manufacturing industry experience, Hands-on with Dataiku, Azure ML, or Snowflake Cortex, Experience with knowledge graphs, AI agents, or multi-agent systems
Candidate Interest Score: 75
Summary: Md. Mostafa expressed interest in the mid role in IT Services & Consulting and is eager to learn more. However, he has concerns about lacking experience in the semiconductor industry and specific tools like Dataiku, Azure ML, or Snowflake Cortex.
⭐ Rahul Mohalder
Match Score: 85
Interest Score: 70
Final Score: 79.0
Status: Consider
Explanation: Matched Skills: Python (scikit-learn, pandas, PyTorch/TensorFlow), SQL, Machine Learning, Data Science
Missing Skills: Semiconductor or manufacturing industry experience, Hands-on with Dataiku, Azure ML, or Snowflake Cortex, 

In [150]:
import pandas as pd

In [151]:
def export_to_csv(final_output, filename="shortlisted_candidates.csv"):

    rows = []

    for c in final_output:
        rows.append({
            "Candidate Name": c.get("candidate_name"),
            "Match Score": c.get("match_score"),
            "Interest Score": c.get("interest_score"),
            "Final Score": c.get("final_score"),
            "Status": c.get("status"),
            "Explanation": c.get("explanation"),
            "Conversation Summary": c.get("conversation_summary")
        })

    df = pd.DataFrame(rows)
    df.to_csv(filename, index=False)

    print(f"✅ CSV exported: {filename}")

In [152]:
export_to_csv(state["final_output"])

✅ CSV exported: shortlisted_candidates.csv
